In [ ]:
!pip install -q -U google-genai pandas

In [ ]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types

# 1. Configuração da sua chave
API_KEY = "# 1. Configuração segura da chave
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)"
client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-3.8-flash"

# 2. Dados de teste (Pergunta, Contexto real e Resposta gerada por um sistema)
eval_dataset = [
    {
        "id": 1,
        "pergunta": "Qual é o prazo para devolução de produtos?",
        "contexto": "Nossa política permite devoluções em até 30 dias corridos após o recebimento, desde que o produto esteja na embalagem original.",
        "resposta_modelo": "Você pode devolver o produto em até 30 dias após receber, se estiver na embalagem original."
    },
    {
        "id": 2,
        "pergunta": "O frete de devolução é gratuito?",
        "contexto": "Nossa política permite devoluções em até 30 dias corridos após o recebimento, desde que o produto esteja na embalagem original.",
        "resposta_modelo": "Sim, o frete de devolução é totalmente gratuito para qualquer região do país."
    },
    {
        "id": 3,
        "pergunta": "Qual o horário de atendimento do suporte?",
        "contexto": "O suporte técnico funciona de segunda a sexta, das 08h às 18h, pelo chat do aplicativo.",
        "resposta_modelo": "O atendimento acontece de segunda a sexta-feira, das 8h às 18h, via chat no app."
    }
]

# 3. Função LLM-as-a-Judge (Avalia fidelidade ao contexto e detecta alucinação)
def avaliar_resposta(pergunta: str, contexto: str, resposta: str) -> dict:
    prompt_juiz = f"""
    Você é um avaliador técnico de sistemas de IA (LLM-as-a-Judge).
    Avalie se a 'Resposta do Modelo' é fiel EXCLUSIVAMENTE ao 'Contexto' fornecido, sem inventar informações (alucinação).

    Pergunta: {pergunta}
    Contexto: {contexto}
    Resposta do Modelo: {resposta}

    Responda APENAS no formato JSON abaixo:
    {{
        "nota_fidelidade": (número inteiro de 1 a 5, onde 5 é totalmente fiel e 1 é alucinação grave),
        "alucinacao_detectada": (true ou false),
        "justificativa": "explicação técnica curta em 1 frase"
    }}
    """

    inicio = time.time()
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt_juiz,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.0
        )
    )
    latencia = round(time.time() - inicio, 2)

    resultado = json.loads(response.text)
    resultado["latencia_segundos"] = latencia
    return resultado

# 4. Execução do Pipeline
resultados_pipeline = []

for item in eval_dataset:
    avaliacao = avaliar_resposta(
        pergunta=item["pergunta"],
        contexto=item["contexto"],
        resposta=item["resposta_modelo"]
    )
    resultados_pipeline.append({
        "ID": item["id"],
        "Pergunta": item["pergunta"],
        "Nota (1-5)": avaliacao["nota_fidelidade"],
        "Alucinação?": avaliacao["alucinacao_detectada"],
        "Latência (s)": avaliacao["latencia_segundos"],
        "Justificativa": avaliacao["justificativa"]
    })

# 5. Exibição e exportação do relatório
df_resultados = pd.DataFrame(resultados_pipeline)
df_resultados.to_csv("relatorio_avaliacao_llm.csv", index=False)
print("=== RELATÓRIO DE AVALIAÇÃO CONCLUÍDO ===")
display(df_resultados)